In [1]:
import lightgbm as lgb
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
from sklearn.metrics import f1_score
from sklearn.model_selection import GridSearchCV, train_test_split

In [2]:
seed = 1234
np.random.seed(seed)

## Data Loading

In [3]:
DATA = pd.read_csv("data/proc_training_nobin.csv")
TEST_DATA = pd.read_csv("data/proc_testing_nobin.csv")

In [4]:
X = DATA.drop(columns='label')
y = DATA['label']

In [5]:
ORI_TEST_DATA = pd.read_csv("data/public_processed.csv")

## Model Construction

In [6]:
lgb_classifier = lgb.LGBMClassifier(
    boosting_type='gbdt',
    objective='binary',
    device='gpu'
)

In [7]:
DATA.columns

Index(['locdt', 'loctm', 'contp', 'etymd', 'conam', 'ecfg', 'insfg', 'iterm',
       'bnsfg', 'stscd', 'ovrlt', 'flbmk', 'hcefg', 'csmcu', 'flg_3dsmk',
       'label', 'fc_ratio', 'mcc_fr', 'region_fr', 'chid_fr'],
      dtype='object')

## Training & Validation

In [8]:
cat_feats = [
    'contp',
    'etymd',
    'ecfg',
    'insfg',
    'bnsfg',
    'stscd',
    'ovrlt',
    'flbmk',
    'hcefg',
    'csmcu',
    'flg_3dsmk',
    # 'mcc_gp',
    # 'region_gp',
    # 'chid_gp'
]

X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.3)

lgb_classifier.fit(
    X_train, y_train,
    categorical_feature=cat_feats,
    callbacks=[lgb.early_stopping(5)],
    eval_set=[(X_valid, y_valid)],
    eval_metric='binary_logloss',
)

[LightGBM] [Info] Number of positive: 22383, number of negative: 6059585
[LightGBM] [Info] This is the GPU trainer!!
[LightGBM] [Info] Total Bins 1439
[LightGBM] [Info] Number of data points in the train set: 6081968, number of used features: 19
[LightGBM] [Info] Using GPU Device: NVIDIA GeForce RTX 4090, Vendor: NVIDIA Corporation
[LightGBM] [Info] Compiling OpenCL Kernel with 256 bins...
[LightGBM] [Info] GPU programs have been built
[LightGBM] [Info] Size of histogram bin entry: 8
[LightGBM] [Info] 8 dense feature groups (46.40 MB) transferred to GPU in 0.033232 secs. 1 sparse feature groups
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.003680 -> initscore=-5.601095
[LightGBM] [Info] Start training from score -5.601095
Training until validation scores don't improve for 5 rounds
Early stopping, best iteration is:
[1]	valid_0's binary_logloss: 0.0146785


LGBMClassifier(device='gpu', objective='binary')

In [9]:
# params = {
#     'num_leaves': [31, 41, 51],
#     'learning_rate': [0.1, 0.2, 0.3],
#     'n_estimators': [2, 50, 100],
# }
# grid_search = GridSearchCV(lgb_classifier, params, cv=14)

# grid_search.fit(
#     data, labels,
#     eval_metric='auc,binary_logloss',
#     categorical_features=cat_feat,
#     callbacks=[lgb.early_stopping(5)]
# )

# print(f'Best params: {grid_search.best_params_}')

In [10]:
y_pred = lgb_classifier.predict(TEST_DATA)
# y_pred = grid_search.predict(TEST_DATA)
res = {
    'txkey': ORI_TEST_DATA['txkey'],
    'label': y_pred
}

pd.DataFrame(res).to_csv('preds.csv', index=False)